In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load Data

In [ ]:
df_awal = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
df_awal

In [ ]:
kolom = [
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    'Churn'
]
df = df_awal[kolom]
df

# Cek Kualitas Data

In [ ]:
# Missing Value
df.isna().sum()

In [ ]:
# duplikat
df.duplicated().sum()

In [ ]:
# Cek Unique
print(df[kolom].nunique())
print('\n'*5)

for c in kolom:
    print(df[c].unique())
    print(df[c].value_counts())
    print()

In [ ]:
# cek hidden value
for i in kolom:
    print(
        (df[i].str.strip() == '').sum()
    )

In [ ]:
# cek konsistensi antar kolom
pd.crosstab(df['PhoneService'], df['MultipleLines'])

In [ ]:
addon = ['OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies']
for c in addon:
    display(pd.crosstab(df['InternetService'], df[c]))

# Analisis Univariat

In [ ]:
# Frekuensi dan Proporsi
for c in kolom:
    tabel = pd.concat([df[c].value_counts(),df[c].value_counts(normalize=True)],axis=1, keys=['jumlah', 'rasio'])
    display(tabel)

In [ ]:
# visualisasi
fig, axes = plt.subplots(3, 3, figsize=(15, 12))

for ax, c in zip(axes.flat, kolom):
    sns.countplot(
        data=df,
        x=c,ax=ax
    )
    ax.set_title(c)
    ax.bar_label(ax.containers[0])
plt.tight_layout()
plt.show()

## Kesimpulan Analisis Univariat

Total data: **7043 pelanggan**.

**1. Layanan Telepon**
- **PhoneService** sangat timpang: 90.3% pelanggan menggunakan layanan telepon, hanya 9.7% yang tidak. Variasi kolom ini kecil sehingga kemungkinan kurang informatif.
- **MultipleLines**: 48.1% tidak menggunakan, 42.2% menggunakan, dan 9.7% tidak memiliki layanan telepon.

**2. Layanan Internet**
- **InternetService**: Fiber optic paling banyak digunakan (44.0%), diikuti DSL (34.4%). Sebanyak 21.7% pelanggan tidak memiliki layanan internet.

**3. Layanan Tambahan (Add-on)**
- Kategori `No internet service` memiliki proporsi yang sama (21.7%) di keenam kolom add-on karena berasal dari kelompok pelanggan yang sama.
- **OnlineSecurity** (28.7%) dan **TechSupport** (29.0%) merupakan add-on yang paling jarang digunakan.
- **OnlineBackup** (34.5%) dan **DeviceProtection** (34.4%) memiliki tingkat penggunaan yang hampir sama.
- **StreamingTV** (38.4%) dan **StreamingMovies** (38.8%) memiliki proporsi Yes dan No yang hampir seimbang.

---

# Ketergantungan Antar Kolom

In [ ]:
tanpa_internet = df[df['InternetService'] == 'No']
print((tanpa_internet[addon] == 'No internet service').all())

In [ ]:
tanpa_phone = df[df['PhoneService'] == 'No']
print((tanpa_phone['MultipleLines'] == 'No phone service').all())

# Interpretasi 

Artinya:

```text
Output pertama: keenam add-on bernilai True. Semua 1526 pelanggan tanpa internet tercatat “No internet service” di semua add-on, tanpa pengecualian.
Output kedua: True. Semua 682 pelanggan tanpa telepon tercatat “No phone service” di MultipleLines.
```

Kesimpulannya, ketergantungannya pasti dan datanya konsisten.

---

# Analisis Bivariat

In [ ]:
df_awal['Churn'].value_counts(normalize=True)

In [ ]:
for c in kolom:
    tabel = pd.crosstab(df[c], df_awal['Churn'], normalize='index')
    display(tabel)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 12))

for ax, c in zip(axes.flat, kolom):
    sns.countplot(
        data=df,
        x=c,
        hue='Churn',
        ax=ax
    )
    ax.set_title(c)
plt.tight_layout()
plt.show()

### Kesimpulan Analisis Bivariat terhadap Churn

Baseline churn rate seluruh pelanggan: **26.5%**.

**1. Pengaruh kuat**
- **InternetService**: churn pelanggan Fiber optic sangat tinggi (41.9%), DSL 19.0%, dan pelanggan tanpa internet hanya 7.4%.
- **OnlineSecurity**: churn pelanggan yang tidak menggunakan 41.8%, sedangkan yang menggunakan hanya 14.6%.
- **TechSupport**: churn pelanggan yang tidak menggunakan 41.6%, sedangkan yang menggunakan hanya 15.2%.

**2. Pengaruh sedang**
- **OnlineBackup**: tidak menggunakan 39.9% vs menggunakan 21.5%.
- **DeviceProtection**: tidak menggunakan 39.1% vs menggunakan 22.5%.

**3. Pengaruh lemah**
- **StreamingTV** (33.5% vs 30.1%) dan **StreamingMovies** (33.7% vs 29.9%): perbedaan churn kecil.
- **MultipleLines**: No 25.0%, Yes 28.6%, No phone service 24.9%, semuanya mendekati baseline.
- **PhoneService**: No 24.9% vs Yes 26.7%, hampir tidak ada perbedaan.

**Catatan:**
- Churn rendah pada kategori `No internet service` (7.4%) disebabkan oleh tidak adanya layanan internet, bukan karena tidak menggunakan add-on (lihat tahap Ketergantungan Antar Kolom).
- Add-on bersifat proteksi/dukungan (OnlineSecurity, TechSupport) memiliki hubungan paling kuat dengan churn yang lebih rendah.
- Hasil ini menunjukkan hubungan, bukan sebab-akibat. Kekuatan hubungan akan diuji secara statistik pada tahap berikutnya.

---

# Uji Statsitik

In [ ]:
from scipy.stats import chi2_contingency
import numpy as np

hasil = []
for c in kolom:
    tabel = pd.crosstab(df[c], df['Churn'])
    chi2, p, dof, _ = chi2_contingency(tabel)
    n = tabel.values.sum()
    v = np.sqrt(chi2 / (n * (min(tabel.shape) - 1)))
    hasil.append([c, round(chi2, 1), p, round(v, 3)])

hasil = pd.DataFrame(hasil, columns=['kolom', 'chi2', 'p_value', 'cramers_v'])
hasil.sort_values('cramers_v', ascending=False)

### Kesimpulan Uji Statistik

Uji Chi-square digunakan untuk menguji hubungan tiap kolom layanan dengan Churn (H0: tidak ada hubungan, α = 0.05). Kekuatan hubungan diukur dengan Cramér's V (< 0.1 lemah, 0.1–0.3 sedang, > 0.3 kuat).

**Ranking kekuatan hubungan dengan Churn:**

| Ranking | Kolom | Cramér's V | Kategori |
|---|---|---|---|
| 1 | OnlineSecurity | 0.347 | Kuat |
| 2 | TechSupport | 0.343 | Kuat |
| 3 | InternetService | 0.322 | Kuat |
| 4 | OnlineBackup | 0.292 | Sedang |
| 5 | DeviceProtection | 0.282 | Sedang |
| 6 | StreamingTV | 0.231 | Sedang |
| 7 | StreamingMovies | 0.231 | Sedang |
| 8 | MultipleLines | 0.040 | Lemah |
| 9 | PhoneService | 0.011 | Lemah, tidak signifikan |

**Temuan:**
- Delapan kolom berhubungan signifikan dengan Churn (p-value < 0.05). Hanya **PhoneService** yang tidak signifikan (p-value = 0.339).
- **MultipleLines** signifikan, tetapi hubungannya sangat lemah (V = 0.040). Signifikansi ini kemungkinan dipengaruhi jumlah data yang besar.
- **OnlineSecurity, TechSupport, dan InternetService** memiliki hubungan terkuat dengan Churn. Hasil ini konsisten dengan analisis bivariat.
- Nilai Cramér's V pada kolom add-on ikut dipengaruhi oleh kategori `No internet service`, yang berasal dari kolom InternetService (lihat tahap Ketergantungan Antar Kolom).

---

# Rangkuman Insight

**Temuan utama:**
1. **Data bersih dan konsisten.** Tidak ada missing value maupun duplikat pada kolom layanan. Kategori `No phone service` dan `No internet service` konsisten dengan kolom layanan utamanya.
2. **Fiber optic adalah layanan internet paling banyak digunakan (44.0%), tetapi juga memiliki churn tertinggi (41.9%).** Angka ini jauh di atas baseline 26.5%.
3. **Add-on proteksi dan dukungan berhubungan paling kuat dengan churn yang lebih rendah.** Pelanggan yang menggunakan OnlineSecurity (churn 14.6%) dan TechSupport (15.2%) jauh lebih jarang churn dibanding yang tidak menggunakan (sekitar 42%). Keduanya memiliki Cramér's V tertinggi (0.347 dan 0.343).
4. **Add-on streaming berpengaruh lemah.** Perbedaan churn antara pengguna dan non-pengguna StreamingTV/StreamingMovies kecil (sekitar 3–4 poin persen).
5. **Layanan telepon hampir tidak berpengaruh.** PhoneService tidak signifikan (p-value = 0.339) dan MultipleLines memiliki hubungan sangat lemah (V = 0.040).

---

**Implikasi untuk preprocessing dan modeling:**
- Kategori `No internet service` dan `No phone service` digabung menjadi `No` untuk menghindari multikolinearitas, karena informasinya sudah terwakili oleh InternetService dan PhoneService.
- Kolom biner di-encode menjadi 0/1. InternetService (3 kategori) di-encode dengan one-hot encoding (`drop_first=True`).
- PhoneService dan MultipleLines dapat dipertimbangkan untuk dihapus atau diuji ulang pengaruhnya pada model, karena hubungannya dengan Churn sangat lemah.
- Target Churn tidak seimbang (26.5% churn), sehingga perlu diperhatikan saat pembagian data dan evaluasi model.

In [ ]:
df_model = df
df_model[kolom] = df_model[kolom].replace({'No internet service': 'No', 'No phone service': 'No'})

In [ ]:
def encode_internet(data):
    data = data.copy()
    data['InternetService_Fiber'] = (data['InternetService'] == 'Fiber optic').astype(int)
    data['InternetService_No'] = (data['InternetService'] == 'No').astype(int)
    return data.drop(columns='InternetService')

df_model = encode_internet(df_model)

In [ ]:
df_model

In [ ]:
df_model.insert(0, 'customerID', df_awal['customerID'])

In [ ]:
df_model.to_csv('seno.csv', index=False)

In [ ]:
df.to_csv('seno1.csv', index=False)